In [1]:
!pip install ultralytics carla


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import carla
import cv2
import numpy as np
import time
import torch
import torch.nn as nn
from torchvision import models, transforms
from ultralytics import YOLO
from PIL import Image
import os

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cuda


In [3]:
class DepthRegressor(nn.Module):
    def __init__(self):
        super(DepthRegressor, self).__init__()
        # Load base architecture
        base_model = models.mobilenet_v3_small(weights=None)
        
        # Assign sub-modules directly to self to match flattened state_dict keys
        self.features = base_model.features
        self.avgpool = base_model.avgpool
        self.classifier = base_model.classifier
        
        # Modify the final layer to output 1 value
        in_features = self.classifier[-1].in_features
        self.classifier[-1] = nn.Linear(in_features, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# Preprocessing transforms
regressor_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
# 1. Load YOLO Models
model_early = YOLO('yolov8m_early_100k.pt')
model_final = YOLO('yolov8m_final_100k.pt')

# 2. Load the Upfront MobileNetV3 Regressor
regressor = DepthRegressor().to(device)
# Use the path where you uploaded your weights on RunPod
weights_path = './uep_regressor_100k.pth' 
regressor.load_state_dict(torch.load(weights_path, map_location=device))
regressor.eval()
print("All models loaded successfully.")

EXIT_THRESHOLD = 0.7236

All models loaded successfully.


/tmp/ipykernel_16035/960551327.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  regressor.load_state_dict(torch.load(weights_path, map_location=device))


In [5]:
client = carla.Client('127.0.0.1', 2000)
client.set_timeout(10.0)
world = client.get_world()
blueprint_library = world.get_blueprint_library()

# Spawn Ego Vehicle
bp = blueprint_library.filter('model3')[0]
spawn_point = world.get_map().get_spawn_points()[0]
vehicle = world.spawn_actor(bp, spawn_point)
vehicle.set_autopilot(True)

# Camera Setup
camera_bp = blueprint_library.find('sensor.camera.rgb')
camera_bp.set_attribute('image_size_x', '640')
camera_bp.set_attribute('image_size_y', '640')
camera_transform = carla.Transform(carla.Location(x=1.5, z=2.4))
camera = world.spawn_actor(camera_bp, camera_transform, attach_to=vehicle)

# Frame storage
current_frame = None

def camera_callback(image):
    global current_frame
    array = np.frombuffer(image.raw_data, dtype=np.dtype("uint8"))
    array = np.reshape(array, (image.height, image.width, 4))
    current_frame = array[:, :, :3]

camera.listen(lambda image: camera_callback(image))

In [6]:
metrics = {'total_latencies': [], 'early_hits': 0, 'final_hits': 0}
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('conditional_output.mp4', fourcc, 20.0, (640, 640))

print("Recording Conditional Inference (100 frames)...")

try:
    for _ in range(100): # Test 100 frames
        if current_frame is None: 
            time.sleep(0.05)
            continue
            
        loop_start = time.perf_counter()
        
        # 1. Regressor Decision
        img_rgb = cv2.cvtColor(current_frame, cv2.COLOR_BGR2RGB)
        input_tensor = regressor_transform(Image.fromarray(img_rgb)).unsqueeze(0).to(device)
        with torch.no_grad():
            difficulty = regressor(input_tensor).item()
        
        # 2. Exit Logic
        if difficulty < EXIT_THRESHOLD:
            results = model_early(current_frame, verbose=False)
            mode = "EARLY"
            metrics['early_hits'] += 1
        else:
            results = model_final(current_frame, verbose=False)
            mode = "FINAL"
            metrics['final_hits'] += 1
            
        total_latency = (time.perf_counter() - loop_start) * 1000
        metrics['total_latencies'].append(total_latency)

        # Draw info and save frame
        annotated = results[0].plot()
        cv2.putText(annotated, f"EXIT: {mode} | Latency: {total_latency:.1f}ms", 
                    (10, 30), 1, 1.2, (0, 255, 0), 2)
        out.write(annotated)

finally:
    out.release()
    print("Conditional Video Saved.")

Recording Conditional Inference (100 frames)...
Conditional Video Saved.


In [7]:
baseline_latencies = []
out_baseline = cv2.VideoWriter('baseline_output.mp4', fourcc, 20.0, (640, 640))

print("Recording Baseline (100 frames)...")

try:
    for _ in range(100):
        if current_frame is None:
            time.sleep(0.05)
            continue
            
        start_time = time.perf_counter()
        results = model_final(current_frame, verbose=False)
        latency = (time.perf_counter() - start_time) * 1000
        baseline_latencies.append(latency)
        
        annotated = results[0].plot()
        cv2.putText(annotated, f"BASELINE | Latency: {latency:.1f}ms", 
                    (10, 30), 1, 1.2, (255, 255, 255), 2)
        out_baseline.write(annotated)

finally:
    out_baseline.release()
    camera.stop()
    vehicle.destroy()
    print("Baseline Video Saved.")

Recording Baseline (100 frames)...
Baseline Video Saved.


In [8]:
avg_cond = np.mean(metrics['total_latencies'])
avg_base = np.mean(baseline_latencies)
savings = avg_base - avg_cond
fps_cond = 1000 / avg_cond
early_rate = (metrics['early_hits'] / 100) * 100

print(f"--- RESULTS ---")
print(f"Baseline Latency:    {avg_base:.2f} ms")
print(f"Conditional Latency: {avg_cond:.2f} ms")
print(f"Net Time Saved:      {savings:.2f} ms/frame")
print(f"System FPS:          {fps_cond:.1f}")
print(f"Early Exit Usage:    {early_rate:.1f}%")

--- RESULTS ---
Baseline Latency:    15.54 ms
Conditional Latency: 73.53 ms
Net Time Saved:      -57.99 ms/frame
System FPS:          13.6
Early Exit Usage:    7.0%
